# KeyClass on synthetic discharge notes: ablation study

**Paper:** Chufan Gao, Mononito Goswami, Jieshi Chen and Artur Dubrawski.
[Classifying Unstructured Clinical Notes via Automatic Weak Supervision](https://arxiv.org/abs/2206.12088v2).
MLHC 2022.

KeyClass assigns diagnostic categories to clinical notes **without any labeled
notes**. It mines frequent phrases, matches each phrase to the closest category
description to create keyword rules, turns rule votes into soft weak labels, and
trains a multilabel classifier on document embeddings.

This notebook runs the whole pipeline with PyHealth's `KeyClass` model and
`pyhealth.weak_supervision` on **synthetic** notes, so it needs no MIMIC access.
It then varies three settings and measures their effect on a held-out synthetic
test set:

1. **n-gram range of the keyword rules** (single words vs phrases). This is the
   ablation proposed for our reproduction; the paper does not report it.
2. **Rule budget** (`top_k` rules per category, and the authors' option to cap
   it at the smallest category).
3. **Input standardization** of embeddings before the classifier.

The last section documents our results on the real MIMIC-III data.

**Setup:** `pip install -e '.[keyclass]'` from the PyHealth repository, then run
all cells (about one minute on a CPU). The first cell downloads NLTK's stopword
list if it is missing.

In [1]:
# Author: Amin Torabi
# Paper title: Classifying Unstructured Clinical Notes via Automatic Weak Supervision
# Paper link: https://arxiv.org/abs/2206.12088v2
# Description: KeyClass ablations (rule n-grams, rule budget, input
# standardization) on synthetic discharge notes.

import warnings

# Set before other imports. NumPy 2.2 with Apple Accelerate reports false
# divide/overflow warnings from correct matrix products, and NLTK's progress bar
# warns at import when ipywidgets is missing.
warnings.filterwarnings("ignore", message=".*encountered in matmul")
warnings.filterwarnings("ignore", message="IProgress not found")

In [2]:
import nltk
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.metrics import roc_auc_score

from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.models import KeyClass
from pyhealth.weak_supervision import (
    KeywordLabeler,
    aggregate_votes,
    clean_note,
    confident_balanced_mask,
)

try:
    nltk.corpus.stopwords.words("english")
except LookupError:
    nltk.download("stopwords", quiet=True)

## Synthetic notes

Six diagnostic categories stand in for the paper's 19 ICD-9 groups. Each has a
short description (the only supervision KeyClass gets) and a bank of phrases
used to write notes. Several phrases share a head word across categories
("heart **failure**", "respiratory **failure**", "renal **failure**"; "chronic
... **disease**"), so a single word is ambiguous where the whole phrase is not.

Each note has one to three true categories, drawn with uneven prevalence like
MIMIC-III. It mixes their phrases with filler sentences, MIMIC-style
de-identification placeholders (`[** ... **]`) and digits. In 30% of notes a
phrase from a category the note does **not** have appears in a negated or
family-history sentence ("ruled out sepsis"), which keyword rules cannot tell
apart from a real diagnosis. True labels are used only for evaluation.

In [3]:
CATEGORIES = {
    "circulatory": dict(
        prevalence=0.70,
        description="heart cardiac circulatory hypertension coronary artery atrial",
        phrases=[
            "heart failure",
            "atrial fibrillation",
            "coronary artery disease",
            "hypertension",
            "cardiac arrest",
            "aortic stenosis",
        ],
    ),
    "respiratory": dict(
        prevalence=0.45,
        description="respiratory lung pulmonary pneumonia airway breathing",
        phrases=[
            "respiratory failure",
            "pneumonia",
            "pulmonary embolism",
            "chronic obstructive pulmonary disease",
            "pleural effusion",
        ],
    ),
    "genitourinary": dict(
        prevalence=0.40,
        description="kidney renal urinary bladder genitourinary",
        phrases=[
            "renal failure",
            "acute kidney injury",
            "urinary tract infection",
            "chronic kidney disease",
            "hydronephrosis",
        ],
    ),
    "infectious": dict(
        prevalence=0.27,
        description="infection sepsis bacterial viral organism",
        phrases=[
            "sepsis",
            "bacteremia",
            "clostridium difficile infection",
            "septic shock",
            "positive blood cultures",
        ],
    ),
    "endocrine": dict(
        prevalence=0.65,
        description="diabetes thyroid metabolic glucose insulin endocrine",
        phrases=[
            "diabetes mellitus",
            "hypothyroidism",
            "hyperlipidemia",
            "insulin dependent diabetes",
            "thyroid nodule",
        ],
    ),
    "mental": dict(
        prevalence=0.30,
        description="depression anxiety psychiatric alcohol substance mental",
        phrases=[
            "major depression",
            "anxiety disorder",
            "alcohol withdrawal",
            "substance abuse",
            "bipolar disorder",
        ],
    ),
}
NAMES = list(CATEGORIES)
FILLER = [
    "patient was admitted to the [**Hospital1 18**] micu on [**2101-3-4**]",
    "vital signs stable on day {n} of admission",
    "follow up with dr [**Last Name (STitle) 123**] in {n} weeks",
    "discharge medications reviewed with the patient and family",
    "physical exam notable for no acute distress",
    "labs were checked daily and electrolytes repleted",
    "patient tolerated a regular diet and ambulated with physical therapy",
]
MENTIONS = [
    "presented with {p}",
    "known history of {p}",
    "treated for {p}",
    "course complicated by {p}",
    "{p} managed medically",
]
NOISE = ["family history of {p}", "ruled out {p}", "no evidence of {p}"]


def make_notes(n: int, seed: int) -> list[dict]:
    # Returns n synthetic notes with text, a patient ID and multi-hot labels.
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        labels = [c for c in NAMES if rng.random() < CATEGORIES[c]["prevalence"]]
        labels = labels or [NAMES[rng.integers(len(NAMES))]]
        picks = rng.choice(len(FILLER), rng.integers(3, 6), replace=False)
        sentences = [FILLER[j].format(n=rng.integers(1, 9)) for j in picks]
        for c in labels:
            bank = CATEGORIES[c]["phrases"]
            for p in rng.choice(bank, rng.integers(1, 4), replace=False):
                template = MENTIONS[rng.integers(len(MENTIONS))]
                sentences.append(template.format(p=p))
        if rng.random() < 0.3:
            others = [c for c in NAMES if c not in labels] or NAMES
            bank = CATEGORIES[others[rng.integers(len(others))]]["phrases"]
            noise = NOISE[rng.integers(len(NOISE))]
            sentences.append(noise.format(p=bank[rng.integers(len(bank))]))
        rng.shuffle(sentences)
        rows.append(
            {
                "patient_id": f"s{seed}-p{i}",
                "text": ". ".join(sentences) + ".",
                "labels": [int(c in labels) for c in NAMES],
            }
        )
    return rows


example = make_notes(600, seed=1)
print(example[0]["text"], "\n")
gold = np.array([r["labels"] for r in example])
print(f"Labels per note: {gold.sum(1).mean():.2f}")
pd.Series(gold.mean(0).round(2), index=NAMES, name="prevalence").to_frame().T

family history of sepsis. course complicated by coronary artery disease. vital signs stable on day 7 of admission. presented with cardiac arrest. physical exam notable for no acute distress. urinary tract infection managed medically. treated for hypothyroidism. known history of hypertension. follow up with dr [**Last Name (STitle) 123**] in 1 weeks. 

Labels per note: 2.86


,circulatory,respiratory,genitourinary,infectious,endocrine,mental
prevalence,0.72,0.44,0.39,0.27,0.69,0.34


## Encoder

The paper embeds phrases, descriptions and notes with BlueBERT. To keep this
notebook offline and fast, a hashed **character n-gram** encoder stands in for
it: phrases that share character sequences with a description score as
similar. It has no clinical knowledge, so the numbers below show how the
pipeline's settings behave, not clinical accuracy. For real notes, use
`pyhealth.weak_supervision.ClinicalTextEncoder` with a BlueBERT checkpoint
instead; nothing else changes.

In [4]:
class CharNgramEncoder:
    # Offline stand-in for BlueBERT: hashed character 3-5 grams.

    def __init__(self, n_features: int = 2**12) -> None:
        self.vectorizer = HashingVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            n_features=n_features,
            alternate_sign=False,
        )

    def encode(self, texts: list[str]) -> np.ndarray:
        return self.vectorizer.transform(list(texts)).toarray().astype(np.float32)

## Pipeline and metrics

`run_keyclass` performs one complete run without gold labels:

1. clean the notes and fit `KeywordLabeler` on the training notes and the six
   descriptions (phrases kept only if they occur in more than 1% of notes);
2. vote with the rules and aggregate votes by majority (`aggregate_votes`);
3. select confident covered notes (`confident_balanced_mask`) and weight each
   note's loss by its weak-label confidence;
4. train PyHealth's `KeyClass` for 20 epochs on the note embeddings;
5. predict a category when its score is at least 1/6 (the uniform share,
   fixed in advance, as in our MIMIC-III runs).

The paper also self-trains the classifier on its own sharpened predictions.
This notebook stops before that step: on the full MIMIC-III run, self-training
collapsed the multilabel predictions (see the last section).

Metrics follow the paper: **precision, recall and F1 averaged over test notes**
(Table 3), plus macro **AUROC** across categories, which measures ranking quality
independently of the threshold. **Weak-label AUROC** scores the rules' own soft
labels against the training notes' true labels, which separates the quality of
the weak supervision from what the classifier learns from it. Each setting is
run on three seeds; each seed draws a new synthetic training and test set.

In [5]:
def sample_prf(pred: np.ndarray, gold: np.ndarray) -> tuple[float, float, float]:
    # Precision, recall and F1 per note, averaged over notes (paper footnote 4).
    hits = (pred & (gold == 1)).sum(1)
    n_pred = pred.sum(1)
    p = np.divide(hits, n_pred, out=np.zeros(len(gold)), where=n_pred > 0)
    r = hits / gold.sum(1)
    f = np.divide(2 * p * r, p + r, out=np.zeros(len(gold)), where=(p + r) > 0)
    return p.mean(), r.mean(), f.mean()


def macro_auroc(scores: np.ndarray, gold: np.ndarray) -> float:
    return float(
        np.mean(
            [
                roc_auc_score(gold[:, c], scores[:, c])
                for c in range(gold.shape[1])
                if 0 < gold[:, c].sum() < len(gold)
            ]
        )
    )


def run_keyclass(
    seed: int,
    ngram_range: tuple[int, int] = (1, 3),
    top_k: int = 30,
    equalize_top_k: bool = False,
    standardize: bool = True,
    epochs: int = 20,
) -> dict:
    # One label-free KeyClass run on a fresh synthetic train/test split.
    train = make_notes(600, seed=2 * seed + 1)
    test = make_notes(300, seed=2 * seed + 2)
    torch.manual_seed(seed)
    encoder = CharNgramEncoder()
    descriptions = {c: CATEGORIES[c]["description"] for c in NAMES}
    train_text = [clean_note(r["text"]) for r in train]
    test_text = [clean_note(r["text"]) for r in test]

    labeler = KeywordLabeler(
        encoder,
        top_k=top_k,
        ngram_range=ngram_range,
        min_df=0.01,
        equalize_top_k=equalize_top_k,
    ).fit(train_text, descriptions)
    weak, covered = aggregate_votes(labeler.transform(train_text), len(NAMES))
    confident = confident_balanced_mask(weak, covered, max_num=len(train))

    x_train, x_test = encoder.encode(train_text), encoder.encode(test_text)
    samples = [
        {
            "patient_id": train[i]["patient_id"],
            "embedding": x_train[i].tolist(),
            "weak_labels": weak[i].tolist(),
            "sample_weight": float(weak[i].max()),
        }
        for i in np.flatnonzero(confident)
    ]
    dataset = create_sample_dataset(
        samples=samples,
        input_schema={"embedding": "tensor", "sample_weight": "tensor"},
        output_schema={"weak_labels": "tensor"},
    )
    model = KeyClass(dataset, input_dim=x_train.shape[1], category_ids=NAMES)
    if standardize:
        model.fit_input_standardization(x_train)
    optimizer = torch.optim.Adam(model.parameters(), lr=1.5e-3)
    loader = get_dataloader(dataset, batch_size=64, shuffle=True)
    for _ in range(epochs):
        model.train()
        for batch in loader:
            optimizer.zero_grad()
            model(**batch)["loss"].backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        scores = model(embedding=torch.as_tensor(x_test))["y_prob"].numpy()
    gold_train = np.array([r["labels"] for r in train])
    gold_test = np.array([r["labels"] for r in test])
    precision, recall, f1 = sample_prf(scores >= 1 / len(NAMES), gold_test)
    return {
        "rules": len(labeler.lfs_),
        "weak_label_auroc": macro_auroc(weak, gold_train),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auroc": macro_auroc(scores, gold_test),
        "labeler": labeler,
    }


def ablate(settings: dict[str, dict], seeds=(0, 1, 2)) -> pd.DataFrame:
    # Runs every setting on every seed; returns mean and std per setting.
    rows = []
    for name, kwargs in settings.items():
        for seed in seeds:
            result = run_keyclass(seed, **kwargs)
            result.pop("labeler")
            rows.append({"setting": name, **result})
    metrics = ["rules", "weak_label_auroc", "precision", "recall", "f1", "auroc"]
    table = pd.DataFrame(rows).groupby("setting", sort=False)[metrics]
    return table.agg(["mean", "std"]).round(3)

### One baseline run

Settings: rules from 1-3 grams, 30 rules per category, standardized embeddings.
The table lists each category's highest-scoring rules.

In [6]:
baseline = run_keyclass(seed=0)
labeler = baseline.pop("labeler")
top_rules = {
    c: [lf.phrase for lf in labeler.lfs_ if lf.category_index == i][:6]
    for i, c in enumerate(NAMES)
}
print({k: round(v, 3) for k, v in baseline.items()})
pd.DataFrame(top_rules)

{'rules': 180, 'weak_label_auroc': 0.896, 'precision': np.float64(0.929), 'recall': np.float64(0.748), 'f1': np.float64(0.801), 'auroc': 0.912}


,circulatory,respiratory,genitourinary,infectious,endocrine,mental
0,coronary artery,respiratory,urinary,infection,insulin dependent diabetes,depression
1,history coronary artery,pulmonary,urinary tract,infection vital,diabetes,substance
2,hypertension,history pulmonary,history urinary,sepsis,thyroid,depression known
3,history hypertension,history pneumonia,treated urinary,tract infection,insulin,major depression
4,hypertension labs,history respiratory,presented urinary,infection known,diabetes labs,depression managed
5,known history hypertension,pneumonia,history urinary tract,infection vital signs,history diabetes,depression patient


## Ablation 1: single words vs phrases as rules

Our proposal's hypothesis: phrases give more specific supervision than single
words, despite covering fewer notes. With `ngram_range=(1, 1)`, a rule like
"failure" must vote for one category even though heart, respiratory and renal
failure belong to three.

In [7]:
ablate(
    {
        "1-grams": dict(ngram_range=(1, 1)),
        "1-2 grams": dict(ngram_range=(1, 2)),
        "1-3 grams (paper)": dict(ngram_range=(1, 3)),
    }
)

rules      weak_label_auroc        precision        recall  \
                    mean  std             mean    std      mean    std   mean   
setting                                                                         
1-grams             93.0  0.0            0.746  0.009     0.514  0.003  0.944   
1-2 grams          180.0  0.0            0.909  0.004     0.763  0.040  0.820   
1-3 grams (paper)  180.0  0.0            0.896  0.010     0.925  0.015  0.743   

                             f1         auroc         
                     std   mean    std   mean    std  
setting                                               
1-grams            0.038  0.640  0.009  0.678  0.015  
1-2 grams          0.016  0.757  0.027  0.908  0.013  
1-3 grams (paper)  0.007  0.793  0.015  0.910  0.009

## Ablation 2: rule budget

`top_k` sets how many rules each category keeps; the authors' default is 300.
Their code can also cap every category at the size of the smallest category's
phrase pool (`equalize_top_k=True`). On a 1,000-note MIMIC-III sample that cap
cut every category to 22 rules and weakened the labels; here we vary the budget
directly.

In [8]:
ablate(
    {
        "top_k=10": dict(top_k=10),
        "top_k=30": dict(top_k=30),
        "top_k=100": dict(top_k=100),
        "top_k=300 (authors' default)": dict(top_k=300),
        "top_k=300, capped at smallest pool": dict(top_k=300, equalize_top_k=True),
    }
)

rules         weak_label_auroc         \
                                      mean     std             mean    std   
setting                                                                      
top_k=10                              60.0   0.000            0.831  0.005   
top_k=30                             180.0   0.000            0.896  0.010   
top_k=100                            600.0   0.000            0.890  0.005   
top_k=300 (authors' default)        1152.0   6.557            0.802  0.006   
top_k=300, capped at smallest pool   764.0  39.950            0.861  0.006   

                                   precision        recall            f1  \
                                        mean    std   mean    std   mean   
setting                                                                    
top_k=10                               0.887  0.019  0.657  0.014  0.722   
top_k=30                               0.925  0.015  0.743  0.007  0.793   
top_k=100                              0.652  0.023  0.839  0.024  0.705   
top_k=300 (authors' default)           0.522  0.006  0.970  0.004  0.651   
top_k=300, capped at smallest pool     0.653  0.028  0.846  0.040  0.710   

                                           auroc         
                                      std   mean    std  
setting                                                  
top_k=10                            0.018  0.856  0.009  
top_k=30                            0.015  0.910  0.009  
top_k=100                           0.007  0.866  0.017  
top_k=300 (authors' default)        0.005  0.751  0.023  
top_k=300, capped at smallest pool  0.008  0.840  0.023

## Ablation 3: input standardization

Mean-pooled BlueBERT vectors of different notes point in almost the same
direction (average cosine similarity about 0.94 on MIMIC-III), and our
classifier trained on them raw gave nearly the same scores to every note.
`KeyClass.fit_input_standardization` stores each dimension's mean and spread
from the unlabeled training embeddings and rescales every input. The paper and
the authors' code feed raw embeddings.

In [9]:
ablate(
    {
        "raw embeddings (paper)": dict(standardize=False),
        "standardized": dict(standardize=True),
    }
)

rules      weak_label_auroc       precision         \
                         mean  std             mean   std      mean    std   
setting                                                                      
raw embeddings (paper)  180.0  0.0            0.896  0.01     0.822  0.024   
standardized            180.0  0.0            0.896  0.01     0.925  0.015   

                       recall            f1         auroc         
                         mean    std   mean    std   mean    std  
setting                                                           
raw embeddings (paper)  0.665  0.022  0.693  0.024  0.801  0.021  
standardized            0.743  0.007  0.793  0.015  0.910  0.009

## Findings on synthetic data

Means over three seeds; each seed is a new synthetic training and test set.

| Ablation | Setting | Weak-label AUROC | Test F1 | Test AUROC |
|---|---|---|---|---|
| Rule n-grams | 1-grams | 0.746 | 0.640 | 0.678 |
| | 1-2 grams | 0.909 | 0.757 | 0.908 |
| | **1-3 grams (paper)** | 0.896 | **0.793** | **0.910** |
| Rule budget | top_k=10 | 0.831 | 0.722 | 0.856 |
| | **top_k=30** | 0.896 | **0.793** | **0.910** |
| | top_k=100 | 0.890 | 0.705 | 0.866 |
| | top_k=300 (authors' default) | 0.802 | 0.651 | 0.751 |
| | top_k=300, capped at smallest pool | 0.861 | 0.710 | 0.840 |
| Input | raw embeddings (paper) | 0.896 | 0.693 | 0.801 |
| | **standardized** | 0.896 | **0.793** | **0.910** |

1. **Phrases beat single words.** With 1-gram rules, words shared across
   categories ("failure", "disease", "chronic") must vote for a single category,
   and weak-label AUROC drops from 0.90 to 0.75; test F1 drops from 0.79 to
   0.64. This supports our proposal's hypothesis. The synthetic notes were
   built with shared head words, so this shows the mechanism rather than its
   size on real notes.
2. **The rule budget has a sweet spot.** Too few rules (10 per category) miss
   notes (recall 0.66); too many add low-quality rules built from filler text,
   and with 300 per category almost every candidate phrase becomes a rule
   (F1 0.65). Capping at the smallest category trims those rules here, but on a
   1,000-note MIMIC-III sample the same cap left only 22 rules per category and
   hurt. The right
   budget depends on how many good phrases each category has, so the authors'
   default of 300 is not a universal choice.
3. **Standardizing embeddings helps the classifier.** The weak labels are
   identical in both rows, yet F1 rises from 0.69 to 0.79 and AUROC from 0.80
   to 0.91, matching what we saw with BlueBERT on MIMIC-III.

These numbers come from a character n-gram encoder and simple synthetic notes.
They show how the settings act on the pipeline, not clinical performance.

## Results on MIMIC-III

We applied the same pipeline to MIMIC-III v1.4 discharge summaries with the 19
ICD-9 categories of the paper's Table 4, using BlueBERT
(`bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12`) embeddings. Data:
52,714 admissions from 41,118 patients, split 75/25 by patient (39,443 training,
13,271 test). Labels: 5.8 categories per admission on average; symptom codes
780-799 have no Table 4 category and were excluded.

**How these numbers were produced.** MIMIC-III requires credentialed PhysioNet
access and may not be redistributed, so these runs cannot be repeated from this
notebook. We ran them offline with our course reproduction scripts, which are
not part of this PR: one builds the cohort and the patient-level split from the
MIMIC-III CSV files, one runs the pipeline, and one computes the metrics and
bootstrap intervals. The pipeline script runs the same steps as this notebook
with a standalone copy of this PR's weak-supervision and model code, and adds
the paper's self-training step. The numbers below are copied from its output
files.

**Full run.** All 39,443 training notes; phrases kept if they occur in more
than 0.1% of notes (the paper's threshold), giving 157,636 candidates and
5,679 rules (up to 300 per category) that cover 99.5% of notes. Majority-vote weak
labels; 6,481 confident notes for initial training; standardized embeddings;
20 epochs; then 98 rounds of self-training. A category is predicted when its
score is at least 1/19, a rule fixed before training. Training took about
19 minutes on one GPU.

**Held-out test set, all 13,271 notes:**

| Method | Precision | Recall | F1 [95% CI] | Labels/note | Macro AUROC |
|---|---|---|---|---|---|
| **KeyClass, ours, before self-training** | 0.436 | 0.660 | **0.493** [0.490, 0.496] | 8.8 | 0.668 |
| KeyClass, ours, after self-training | 0.652 | 0.179 | 0.260 [0.257, 0.264] | 1.5 | 0.650 |
| KeyClass, ours, 1,000-note sample, after self-training | 0.353 | 0.754 | 0.456 [0.454, 0.458] | 12.4 | 0.610 |
| Categories with training prevalence of at least 50% (uses gold labels) | 0.663 | 0.381 | 0.456 [0.453, 0.460] | 3.0 | - |
| KeyClass, paper Table 3 | 0.507 | 0.896 | 0.625 | - | - |

1. **More training data helps.** Going from 1,000 to 39,443 training notes
   raised F1 from 0.456 to 0.493 and AUROC from 0.61 to 0.67. Before
   self-training, the model beats the prevalence baseline, whose intervals do
   not overlap with ours, but stays below the paper's 0.625.
2. **Self-training collapsed the multilabel predictions.** Following the paper,
   self-training sharpens each note's scores as one distribution over
   categories. Repeated for 98 rounds, it moved almost all of each note's
   weight onto one category: on training notes the median top score reached
   0.98, and test predictions fell to 1.5 categories per note (true notes have 5.8), and recall dropped
   from 0.66 to 0.18. With only 2 rounds on the 1,000-note sample the effect
   did not show. The paper's self-training results (its Table 6) are on
   single-label datasets; for this multilabel task we report the model before
   self-training as our main result. We did not tune the number of rounds,
   because choosing it from test results would leak test information.
3. **Per category** (before self-training, against the paper's Table 4), we do
   better on neoplasms (F1 0.32 vs 0.03), perinatal conditions (0.87 vs 0.00)
   and infectious diseases (0.50 vs 0.49), and worse on circulatory (0.70 vs
   0.92), endocrine (0.71 vs 0.86), supplementary V codes (0.69 vs 0.83) and
   external causes (0.52 vs 0.61).

**Diagnostics on a 1,000-note training sample** (gold labels used only to
measure, before the full run):

- Capping the rule budget at the smallest category left 22 rules per category,
  because in BlueBERT space External causes won only 22 phrases. Removing the
  cap (300 per category) raised weak-label AUROC from 0.57 to 0.70. On the full
  training set every category reached about 300 rules, and External causes went
  from F1 0.00 on the sample to 0.52.
- Standardizing embeddings raised classifier F1 from 0.37 to 0.47 (mean of
  three seeds) and made results stable across seeds.

**Not done:** the 1-gram vs 1-3-gram ablation on MIMIC-III. The n-gram
comparison in this notebook is on synthetic notes only.